# Reactive agents with OPC UA integration (Mesa 3.0.x)

This notebook connects two worlds:

- the **agent-based simulation** with Mesa 3.0.x,
- **industrial communication** via OPC UA (asyncua).

## Learning objectives

After this notebook you can:

1. explain why Mesa simulation and OPC UA communication are operated **decoupled**,
2. use a **shared data buffer** between synchronous simulation code and asynchronous network code,
3. extend Mesa agents so that they **write state data to an OPC UA server**,
4. run the OPC UA HOT monitor **in parallel** to the running simulation.

## Overall architecture

```
┌─────────────────────────────────────────────────────────┐
│                   Jupyter process                        │
│                                                          │
│  Mesa simulation (synchronous)                           │
│  ┌─────────────────────────────┐                         │
│  │ MachineAgent.step()         │                         │
│  │   → writes to               │                         │
│  │     opcua_write_buffer      │──────┐                  │
│  └─────────────────────────────┘      │                  │
│                                       ▼                  │
│  asyncio event loop (parallel)  ┌─────────────────────┐  │
│  ┌──────────────────────┐       │ opcua_write_buffer   │  │
│  │ opcua_writer()       │◄──────│ {"M01": {Temp, State,│  │
│  │   → write_value()    │       │          Busy}, ...} │  │
│  └──────────────────────┘       └─────────────────────┘  │
│  ┌──────────────────────┐                                 │
│  │ subscription_hot_    │                                 │
│  │ monitor()            │                                 │
│  │   → reads from server│                                 │
│  │   → sets RepairNeeded│                                 │
│  └──────────────────────┘                                 │
│                    │                                      │
└────────────────────┼──────────────────────────────────────┘
                     │ OPC UA (TCP)
                     ▼
        ┌─────────────────────────┐
        │  OPC UA factory server  │
        │  M01/Temperature        │
        │  M01/State              │
        │  M01/Busy               │
        │  M01_RepairNeeded       │
        │  ...                    │
        └─────────────────────────┘
```

**Why a buffer instead of direct OPC UA calls in the agent?**

`MachineAgent.step()` runs synchronously in the Mesa scheduler. OPC UA write operations
are `async` and cannot be called directly from synchronous code.
The buffer (`opcua_write_buffer`) cleanly decouples both worlds:
Mesa writes values into it, an asyncio task reads them out and transfers
them to the server – without blocking each other.


## Task: connect Mesa to OPC UA

In this student version **three places** are missing; they are marked with `___` and a `# TODO`:

1. **TODO 1 – `MachineAgent.step()`:** write temperature (rounded to 2 decimals), state and busy flag into the `opcua_write_buffer`.
2. **TODO 2 – `MaintenanceAgent.step()`:** reset the `RepairNeeded` flag in the `opcua_repair_buffer` after the repair.
3. **TODO 3 – `opcua_writer()`:** write the values from the buffer to the three OPC UA nodes `Temperature`, `State` and `Busy`.

**Prerequisite:** the OPC UA server from `01_OPC_Server_solution.ipynb` is running. The sample solution is in `04_Reactive_Agents_OPC_solution.ipynb`.


## 1. Imports and output widget

First we set up the `Output` widget and redirect `print` globally
so that OPC UA output from asyncio callbacks becomes visible.


In [ ]:
import asyncio
import builtins
import ipywidgets as widgets
from IPython.display import display

# The OPC UA HOT monitor client must be available as OPC_Hot_Monitor_Client_en.py
# in the same directory.
import OPC_Hot_Monitor_Client_en as hot_client

# ── Output widget for OPC UA messages ─────────────────────────────────────────
# All print() calls – including those from asyncio callbacks – end up here.
opcua_out = widgets.Output(layout=widgets.Layout(
    height="150px",
    overflow_y="auto",
    border="1px solid gray",
))

_original_print = builtins.print
def _patched_print(*args, **kwargs):
    with opcua_out:
        _original_print(*args, **kwargs)
builtins.print = _patched_print

print("Mesa + OPC UA environment imports ready.")


## 2. Shared data buffer

The `opcua_write_buffer` is the central link between Mesa and OPC UA.

- **Mesa writes** the current agent values into it after every `step()`.
- **The asyncio writer task reads** the buffer periodically and transfers the values
  to the OPC UA server.

Since Mesa and the writer task run in the same Python process and the asyncio
event loop is cooperative (no real multithreading), no locks are
required – there are no race conditions.

```
MachineAgent.step()          asyncio writer task
      │                             │
      │  opcua_write_buffer         │
      │  ┌─────────────────────┐    │
      └─►│ "M01": {42.3, "WARM"│──► │ write_value() → OPC UA server
         │ "M02": {71.1, "HOT" │    │
         └─────────────────────┘    │
```


In [ ]:
# Shared data buffer: machine name → current state values
# Written by MachineAgent.step() and read by opcua_writer().
opcua_write_buffer: dict = {}

# Buffer for RepairNeeded flags that are to be reset
# key = machine name, value = False (reset RepairNeeded)
opcua_repair_buffer: dict = {}

## 3. Mesa agents with OPC UA connection

Compared with the original version there are two extensions:

### 3.1 MachineAgent: `machine_name` and buffer update

Each `MachineAgent` gets a fixed **OPC UA machine name** (e.g. `"M01"`)
that corresponds to the node on the server. At the end of every `step()` it writes
its current values into the `opcua_write_buffer`.

**Why at the end of `step()` and not directly to the server?**
OPC UA write operations are `async` – they cannot be called directly from synchronous
Mesa code. The buffer is the clean decoupling.

### 3.2 FactoryModelExtended: numbering the machines

So that each agent has a unique server name, the machines are numbered when
they are created: `M01`, `M02`, ...


In [ ]:
from mesa import Agent
import random


class MachineAgent(Agent):
    """Reactive machine agent with OPC UA connection via a shared data buffer.

    Extensions compared with the basic version:
    - machine_name: unique name of the OPC UA node (e.g. 'M01')
    - step() writes temperature, state and busy flag to opcua_write_buffer
    """

    def __init__(self, model, threshold=70, machine_name=None):
        super().__init__(model)
        self.temperature = 20.0
        self.threshold = threshold
        self.state = "OK"
        self.busy = False
        self.machine_name = machine_name  # ← OPC UA node name, e.g. "M01"

    def sense_temperature(self):
        if self.state != "HOT":
            self.temperature += random.uniform(-1, 2)

    def decide(self):
        if self.temperature > self.threshold:
            self.state = "HOT"
        elif self.temperature > 0.75 * self.threshold:
            self.state = "WARM"
        elif self.temperature < 20:
            self.state = "COOL"
        else:
            self.state = "OK"

    def act(self):
        pass  # cooling is done by the MaintenanceAgent

    def step(self):
        self.sense_temperature()
        self.decide()
        self.act()
        # ── OPC UA: write state to buffer ─────────────────────────────────
        # The asyncio writer task periodically transfers the buffer to the server.
        # TODO 1: Write temperature (rounded to 2 decimals), state and busy flag
        #         into the buffer under the machine name.
        if self.machine_name:
            opcua_write_buffer[self.machine_name] = {
                "Temperature": ___,
                "State":       ___,
                "Busy":        ___,
            }


class MaintenanceAgent(Agent):
    """Moves through the grid and repairs HOT machines."""

    def step(self):
        neighbours = self.model.grid.get_neighborhood(
            self.pos, moore=True, include_center=True
        )
        self.model.grid.move_agent(self, random.choice(neighbours))

        for agent in self.model.grid.get_cell_list_contents([self.pos]):
            if isinstance(agent, MachineAgent) and agent.state == "HOT":
                agent.temperature = agent.threshold - 40
                agent.state = "OK"
                agent.busy = False
                # ── OPC UA: reset RepairNeeded ────────────────────────────
                if agent.machine_name:
                    opcua_repair_buffer[agent.machine_name] = ___  # TODO 2: Which value resets the flag?


class OrderAgent(Agent):
    """Production order: looks for a free machine and occupies it for 'duration' steps."""

    def __init__(self, model, duration=10):
        super().__init__(model)
        self.duration = duration
        self.assigned_machine = None

    def step(self):
        # Case 1: no machine yet → search and assign
        if self.assigned_machine is None:
            neighbours = self.model.grid.get_neighborhood(
                self.pos, moore=True, include_center=True
            )
            self.model.grid.move_agent(self, random.choice(neighbours))

            for agent in self.model.grid.get_cell_list_contents([self.pos]):
                if isinstance(agent, MachineAgent) and agent.state != "HOT" and not agent.busy:
                    agent.busy = True
                    agent.temperature += 5
                    self.assigned_machine = agent
                    break

        # Case 2: machine occupied → count down the duration
        else:
            self.duration -= 1
            for agent in self.model.grid.get_cell_list_contents([self.pos]):
                if isinstance(agent, MachineAgent) and agent.state != "HOT":
                    agent.temperature += 5
            if self.duration <= 0:
                self.assigned_machine.busy = False
                self.model.grid.remove_agent(self)
                self.remove()


In [ ]:
from mesa import Model
from mesa.space import MultiGrid
from mesa.datacollection import DataCollector


def count_hot_machines(model):
    return sum(1 for a in model.agents if isinstance(a, MachineAgent) and a.state == "HOT")

def count_warm_machines(model):
    return sum(1 for a in model.agents if isinstance(a, MachineAgent) and a.state == "WARM")

def count_orders(model):
    return sum(1 for a in model.agents if isinstance(a, OrderAgent))


class FactoryModelExtended(Model):
    """Extended factory model with machine, maintenance and order agents.

    Extension: machines receive unique names (M01, M02, ...)
    that correspond to the OPC UA nodes on the server.
    """

    def __init__(self, width=10, height=10, density=0.3, threshold=70,
                 n_maintenance=2, n_orders=5, seed=None):
        super().__init__(seed=seed)
        self.width = width
        self.height = height
        self.grid = MultiGrid(width, height, torus=False)

        # Place order agents first (lower unique_ids)
        for _ in range(n_orders):
            pos = (self.random.randrange(width), self.random.randrange(height))
            self.grid.place_agent(OrderAgent(self), pos)

        # Number machine agents with OPC UA names
        machine_counter = 1
        for x in range(width):
            for y in range(height):
                if self.random.random() < density:
                    name = f"M{machine_counter:02d}"
                    self.grid.place_agent(
                        MachineAgent(self, threshold=threshold, machine_name=name), (x, y)
                    )
                    machine_counter += 1

        for _ in range(n_maintenance):
            pos = (self.random.randrange(width), self.random.randrange(height))
            self.grid.place_agent(MaintenanceAgent(self), pos)

        self.datacollector = DataCollector(
            model_reporters={
                "HotMachines":  count_hot_machines,
                "WarmMachines": count_warm_machines,
                "Orders":       count_orders,
            }
        )
        self.datacollector.collect(self)

    def step(self):
        self.agents.shuffle_do("step")
        self.datacollector.collect(self)


## 4. Visualisation


In [ ]:
from mesa.visualization import SolaraViz, make_plot_component, make_space_component

DENSITY      = 0.2
THRESHOLD    = 70
N_MAINTENANCE = 2
N_ORDERS     = 30


def agent_portrayal_ext(agent):
    if isinstance(agent, MachineAgent):
        return {
            "color": {
                "HOT":  "red",
                "WARM": "orange",
                "COOL": "skyblue",
                "OK":   "green",
            }.get(agent.state, "gray"),
            "size": 600, "marker": "s", "alpha": 0.7,
        }
    if isinstance(agent, MaintenanceAgent):
        return {"color": "cyan",  "size": 300, "marker": "o", "alpha": 0.95}
    if isinstance(agent, OrderAgent):
        return {"color": "black", "size": 200, "marker": "s", "alpha": 0.85}
    return {}


def make_post_process_ext(model):
    def post_process(ax):
        ax.figure.set_size_inches(6, 6)
        legend = ax.get_legend()
        if legend:
            legend.remove()
        for txt in ax.texts[:]:
            txt.remove()
        for agent in model.agents:
            x, y = agent.pos
            if isinstance(agent, MachineAgent):
                # show OPC UA name and temperature
                label = f"{agent.machine_name}\n{agent.temperature:.1f}°C" if agent.machine_name else f"{agent.temperature:.1f}°C"
                ax.text(x, y, label,
                        ha="center", va="center",
                        fontsize=5, color="white", fontweight="bold")
            elif isinstance(agent, MaintenanceAgent):
                ax.text(x, y, "W",
                        ha="center", va="center",
                        fontsize=9, color="black", fontweight="bold")
            elif isinstance(agent, OrderAgent):
                ax.text(x, y, str(agent.unique_id),
                        ha="center", va="center",
                        fontsize=6, color="white")
        ax.grid(True, alpha=0.3)
    return post_process


model_ext = FactoryModelExtended(
    width=10, height=10, density=DENSITY, threshold=THRESHOLD,
    n_maintenance=N_MAINTENANCE, n_orders=N_ORDERS,
)

space_ext = make_space_component(
    agent_portrayal_ext,
    post_process=make_post_process_ext(model_ext),
)
plot_ext = make_plot_component(
    {"HotMachines": "red", "WarmMachines": "orange", "Orders": "black"}
)


## 5. OPC UA HOT monitor (reading from the server)

The HOT monitor reads temperatures from the OPC UA server and sets
`Mxx_RepairNeeded = True` as soon as a machine becomes HOT.

It runs as an asyncio task in parallel to the simulation – both share
the Jupyter event loop but do not block each other.


In [ ]:
hot_monitor_task = None

def start_hot_monitor_in_background(
    mode: str = "subscription",
    runtime_seconds: float = 300.0,
    interval: float = 1.0,
    publishing_interval_ms: int = 500,
):
    """Start the OPC UA HOT monitor as an asyncio background task."""
    global hot_monitor_task
    loop = asyncio.get_event_loop()

    if mode == "polling":
        coro = hot_client.polling_hot_monitor(
            runtime_seconds=runtime_seconds,
            poll_interval=interval,
        )
    else:
        coro = hot_client.subscription_hot_monitor(
            runtime_seconds=runtime_seconds,
            publishing_interval_ms=publishing_interval_ms,
        )

    hot_monitor_task = loop.create_task(coro)
    print(f"HOT monitor started ({mode} mode). Task:", hot_monitor_task)


def cancel_hot_monitor():
    """Cancel the running HOT monitor task."""
    global hot_monitor_task
    if hot_monitor_task is not None and not hot_monitor_task.done():
        hot_monitor_task.cancel()
        print("HOT monitor task cancelled.")
    else:
        print("No running HOT monitor task found.")


## 6. OPC UA writer task (writing to the server)

This task is the heart of the new coupling. It:

1. **connects once** to the OPC UA server,
2. **resolves all machine nodes once** and stores them in the `node_cache`
   (this is much more efficient than calling `get_child()` for every write),
3. **periodically reads** the `opcua_write_buffer` and writes the values to the server.

### Why node caching?

`get_child()` is a network call – on every write operation it would
increase latency unnecessarily. By resolving the nodes once at start-up
we reduce each write cycle to plain `write_value()` calls.

### Address space mapping

The OPC UA server creates machines under the following path:

```
Objects / Factory / Machines / M01 / Temperature
                                   / State
                                   / Busy
```

The namespace index `nsidx` is queried dynamically from the server –
no hard-coded indices.


In [ ]:
from asyncua import Client

OPC_UA_URL    = "opc.tcp://localhost:4840/freeopcua/server/"
OPC_UA_NS_URI = "http://ostfalia.de/ipt/factory"

opcua_writer_task = None  # global handle for cancelling


async def opcua_writer(runtime_seconds: float = 300.0, interval: float = 1.0):
    """Read opcua_write_buffer and write temperature, state and busy flag to the OPC UA server.

    Procedure:
        1. Connect and determine the namespace index.
        2. Resolve all machine nodes once and cache them.
        3. Write loop: read buffer → write_value() → wait.

    Parameters
    ----------
    runtime_seconds : float
        Runtime of the writer in seconds.
    interval : float
        Pause between two write cycles in seconds.
    """
    async with Client(url=OPC_UA_URL) as client:
        nsidx = await client.get_namespace_index(OPC_UA_NS_URI)
        print(f"OPC UA writer connected. Namespace index: {nsidx}")

        # ── Step 1: Resolve all machine nodes once ────────────────────────
        # We try M01…M50; nodes that do not exist are skipped.
        node_cache: dict = {}
        for i in range(1, 51):
            name = f"M{i:02d}"
            base = f"0:Objects/{nsidx}:Factory/{nsidx}:Machines/{nsidx}:{name}"
            try:
                node_cache[name] = {
                    "Temperature": await client.nodes.root.get_child(
                        f"{base}/{nsidx}:Temperature"),
                    "State":       await client.nodes.root.get_child(
                        f"{base}/{nsidx}:State"),
                    "Busy":        await client.nodes.root.get_child(
                        f"{base}/{nsidx}:Busy"),
                }
            except Exception:
                break  # No further machines on the server

        print(f"OPC UA writer: {len(node_cache)} machine nodes cached ({', '.join(node_cache)}).")

        # ── Cache RepairNeeded nodes ─────────────────────────────────
        # Pfad: Objects/Factory/Maintenance/Jobs/Mxx_RepairNeeded
        repair_cache: dict = {}
        jobs_base = f"0:Objects/{nsidx}:Factory/{nsidx}:Maintenance/{nsidx}:Jobs"
        for name in node_cache:
            try:
                repair_cache[name] = await client.nodes.root.get_child(
                    f"{jobs_base}/{nsidx}:{name}_RepairNeeded"
                )
            except Exception as e:
                print(f"RepairNeeded node for {name} not found: {e}")

        print(f"OPC UA writer: {len(repair_cache)} RepairNeeded nodes cached.")

        # ── Set all RepairNeeded flags to False at start ──────────────
        for name, node in repair_cache.items():
            try:
                await node.write_value(False)
            except Exception as e:
                print(f"Initialisation error {name}_RepairNeeded: {e}")
        print(f"All {len(repair_cache)} RepairNeeded flags initialised (→ False).")
    

        # ── Step 2: Write loop ─────────────────────────────────────────────
        deadline = asyncio.get_event_loop().time() + runtime_seconds
        while asyncio.get_event_loop().time() < deadline:
            for machine_name, values in list(opcua_write_buffer.items()):
                if machine_name not in node_cache:
                    continue  # machine does not exist on the server
                nodes = node_cache[machine_name]
                try:
                    # TODO 3: Write the values from the buffer (values) to the three OPC UA nodes.
                    await nodes["Temperature"].write_value(___)
                    await nodes["State"].write_value(___)
                    await nodes["Busy"].write_value(___)
                except Exception as e:
                    print(f"Write error {machine_name}: {e}")

            await asyncio.sleep(interval)

            # ── NEW: reset RepairNeeded flags ─────────────────────────────
            for machine_name, value in list(opcua_repair_buffer.items()):
                if machine_name not in repair_cache:
                    continue
                try:
                    await repair_cache[machine_name].write_value(value)
                    del opcua_repair_buffer[machine_name]  # ← write once, then delete
                    print(f"{machine_name}_RepairNeeded → {value}")
                except Exception as e:
                    print(f"RepairNeeded write error {machine_name}: {e}")
            
        print("OPC UA writer finished.")


def start_opcua_writer(runtime_seconds: float = 300.0, interval: float = 1.0):
    """Start the OPC UA writer as an asyncio background task."""
    global opcua_writer_task
    loop = asyncio.get_event_loop()
    opcua_writer_task = loop.create_task(
        opcua_writer(runtime_seconds=runtime_seconds, interval=interval)
    )
    print(f"OPC UA writer started (interval: {interval}s). Task:", opcua_writer_task)


def cancel_opcua_writer():
    """Cancel the running writer task."""
    global opcua_writer_task
    if opcua_writer_task is not None and not opcua_writer_task.done():
        opcua_writer_task.cancel()
        print("OPC UA writer task cancelled.")
    else:
        print("No running writer task found.")


## 7. Starting simulation, HOT monitor and writer together

Now three components run in parallel in the same Jupyter process:

| Component | Type | Task |
|---|---|---|
| `SolaraViz` | synchronous (main thread) | visualise the Mesa simulation |
| `subscription_hot_monitor` | asyncio task | read temperatures from the server, set RepairNeeded |
| `opcua_writer` | asyncio task | write the buffer to the server |

**Order:** First start the writer and monitor, then show `page_ext` as the
last cell output – otherwise SolaraViz blocks the event loop before
the tasks are registered.


In [ ]:
# Start the OPC UA writer (writes Mesa states to the server)
start_opcua_writer(runtime_seconds=300, interval=1.0)

# Start the HOT monitor (reads from the server, sets RepairNeeded flags)
start_hot_monitor_in_background(mode="subscription", runtime_seconds=300)

# Start the Mesa simulation
page_ext = SolaraViz(
    model_ext,
    components=[space_ext, plot_ext],
    name="Extended factory model with OPC UA",
    play_interval=200,
)


### Show OPC UA output


In [ ]:
display(opcua_out)

In [ ]:
page_ext

## 8. Stopping the tasks

With the following cells you can cancel the background tasks manually at any time –
e.g. to change parameters and restart.


In [ ]:
# cancel_hot_monitor()
# cancel_opcua_writer()


## 9. Summary and possible extensions

### What was implemented?

- **MachineAgent** writes temperature, state and busy flag to the `opcua_write_buffer`
  after every `step()`.
- **`opcua_writer()`** periodically transfers the buffer to the OPC UA server –
  completely decoupled from the Mesa scheduler.
- **HOT monitor** reads temperatures from the server and sets `RepairNeeded` flags.

### Architecture principle: decoupling of reading and writing

```
Mesa (synchronous)       Buffer          OPC UA (async)
─────────────────        ──────          ──────────────
step() → write      ──►  dict    ──►  write_value()
                                  ◄──  read_value() (HOT monitor)
```

### Possible next steps

1. **Read RepairNeeded flags**: the `MaintenanceAgent` could read `Mxx_RepairNeeded`
   from the server and navigate specifically to overheated machines.

2. **Coordinate orders via OPC UA**: the `OrderAgent` could look for free machines
   via the server address space instead of the local grid.

3. **Adjust the write interval**: for real plants `interval=2.0` is often sufficient –
   this reduces network load considerably.

4. **Encapsulate the writer in a Mesa agent**: an `OpcUaWriterAgent` could
   empty the buffer in its `step()` – then the writer is fully integrated into the
   agent system.
